## 02 - Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('Data/combined_sold.csv')

/var/folders/jn/2yd4xw3j43l6tt2w0gfb6nh80000gn/T/ipykernel_92169/69749230.py:1: DtypeWarning: Columns (0,1,4,9,78,79,80,81,82) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Data/combined_sold.csv')


In [3]:
nums_before = df.size
nums_before

25395888

In [4]:
df.isnull().sum()

BuyerAgentAOR                   37349
ListAgentAOR                    33976
Flooring                       107944
ViewYN                          27063
WaterfrontYN                   302202
                                ...  
lonfilled                      254434
OriginatingSystemName          260912
OriginatingSystemSubName       260912
BuyerAgencyCompensationType    268392
BuyerAgencyCompensation        268404
Length: 84, dtype: int64

In [5]:
# removing columns that have over 80% missing/null values

def get_nan_cols(df, thresh=0.8):
    threshold = len(df.index) * thresh
    return [c for c in df.columns if df[c].isnull().sum() >= threshold]

print(f"Columns With Over 80% Missing Values: {get_nan_cols(df)}")

df = df.dropna(axis=1, thresh=0.2*len(df))

Columns With Over 80% Missing Values: ['WaterfrontYN', 'BasementYN', 'FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'ElementarySchool', 'BuilderName', 'TaxYear', 'BuildingAreaTotal', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'HighSchool', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict', 'latfilled', 'lonfilled', 'OriginatingSystemName', 'OriginatingSystemSubName', 'BuyerAgencyCompensationType', 'BuyerAgencyCompensation']


In [6]:
# drop these 2 columns since they have no more predictive power after filitering to single family homes
df = df.drop(columns=['PropertyType', 'PropertySubType'])


# drop columns that are solely used for identification and/or have no predictive power
df = df.drop(columns = ['ListAgentFullName', 'ListAgentFirstName', 'ListAgentLastName','ListAgentEmail', 
                        'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'BuyerAgentMlsId',
                        'BuyerAgentAOR', 'ListAgentAOR', 'BuyerOfficeAOR', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
                        'ListingKey', 'ListingId', 'ListingKeyNumeric', 'AssociationFeeFrequency', 'OriginalListPrice', 'ListPrice',
                        'DaysOnMarket', 'ContractStatusChangeDate', 'PurchaseContractDate' ])

df = df.drop(columns = [])


In [7]:
df.columns

Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice',
       'Latitude', 'Longitude', 'UnparsedAddress', 'LivingArea',
       'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'AttachedGarageYN',
       'ParkingTotal', 'LotSizeAcres', 'SubdivisionName', 'YearBuilt',
       'StreetNumberNumeric', 'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet'],
      dtype='object')

In [8]:
df.isnull().sum()

Flooring                 107944
ViewYN                    27063
PoolPrivateYN             29314
CloseDate                     0
ClosePrice                    2
Latitude                  11174
Longitude                 11174
UnparsedAddress             287
LivingArea                  157
MLSAreaMajor              43099
CountyOrParish                0
MlsStatus                     0
AttachedGarageYN          35107
ParkingTotal                344
LotSizeAcres               5214
SubdivisionName          197527
YearBuilt                   218
StreetNumberNumeric         355
BathroomsTotalInteger        50
City                        243
BedroomsTotal                 0
ListingContractDate           0
StateOrProvince               0
FireplaceYN                 198
Stories                   38679
Levels                    28776
LotSizeArea                5164
MainLevelBedrooms        122069
NewConstructionYN         21958
GarageSpaces              11217
HighSchoolDistrict        78727
PostalCo

In [ ]:
# since theres only 2 rows with missing ClosePrice and 1 row with missing PostalCode, drop those rows
df = df.dropna(subset=['ClosePrice', 'PostalCode'])

#filling in some null values
#cols = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeArea','LotSizeAcres', 'LotSizeSquareFeet', 'MainLevelBedrooms', 'ParkingTotal']
#for col in cols:
#    median = df[col].median()
#    df[col] = df[col].fillna(median)

cols = ['Flooring', 'UnparsedAddress', 'MLSAreaMajor', 'SubdivisionName', 'HighSchoolDistrict', 'Levels', 'City']
for col in cols:
    df[col] = df[col].fillna('Unknown')


df['AssociationFee'] = df['AssociationFee'].fillna(0)
df['GarageSpaces'] = df['GarageSpaces'].fillna(0)

# assume nulls means 1 stories
df['Stories'] = df['Stories'].fillna(1.0)

In [10]:
# removing invalid rows

df['neg_closeprice_flag'] = df['ClosePrice'] <= 0
df['neg_livingarea_flag'] = df['LivingArea'] <= 0
df['neg_bedbath_flag'] = (df['BedroomsTotal'] < 0) | (df['BathroomsTotalInteger'] < 0)

df = df[~df['neg_livingarea_flag']]
df = df[~df['neg_bedbath_flag']]
df = df[~df['neg_closeprice_flag']]

df = df.drop(columns=['neg_closeprice_flag', 'neg_livingarea_flag', 'neg_bedbath_flag' ])

In [11]:
df.to_csv('Data/cleaned_sold.csv', index=False)

In [12]:
nums_after = df.size
nums_after

print(f'Number of Rows Before Processing: {nums_before}')
print(f'Number of Rows After Processing: {nums_after}')
print(f'Number of Rows Removed: {nums_before - nums_after}')

Number of Rows Before Processing: 25395888
Number of Rows After Processing: 10275106
Number of Rows Removed: 15120782


### Create Train/Test Split + Preprocessing

In [13]:
from utils import create_time_split, get_preprocessing_pipeline

df = pd.read_csv('Data/cleaned_sold.csv')
df['CloseDate'] = pd.to_datetime(df['CloseDate'])

max_date = df['CloseDate'].max()
test_start_date = max_date - pd.DateOffset(months=1)

train_df, test_df = create_time_split(df, 'CloseDate', 12, test_start_date, max_date)

X_train = train_df.drop(columns=['ClosePrice'])
y_train = train_df['ClosePrice']

X_test = test_df.drop(columns=['ClosePrice'])
y_test = test_df['ClosePrice']

preprocessor = get_preprocessing_pipeline()

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

Training Window (X=12 months): 2025-03-30 to 2026-03-30 | Rows: 122233
Testing Window (1 month): 2026-03-30 to 2026-04-30
